# Desafio Técnico — BMC Power BI Crawler

**Junior Data Engineer | Crawler de dados de commodities da Bolsa Mercantil de Colombia**

---

## Objetivo

Extrair dados de commodities do dashboard Power BI público da Bolsa Mercantil
de Colombia (BMC), reproduzindo o comportamento que o browser do iframe faz
internamente — mas usando apenas Python e `requests`, **sem automação de browser**.

## Estratégia

O dashboard é renderizado por um iframe Power BI Embedded. O autor do desafio
sinaliza explicitamente que o exercício avalia a **capacidade de inspecionar
tráfego de rede, identificar endpoints, e realizar engenharia reversa do payload**.
A solução implementa esse fluxo em três etapas:

| Etapa | Endpoint | Verbo | O que retorna |
|---|---|---|---|
| 1 | AWS Gateway BMC | GET | EmbedToken (curto, abre a sessão) |
| 2 | `modelsAndExploration` | GET | MWCToken + capacityUri + bootstrap completo |
| 3 | `{capacityUri}/query` | POST | Dados em formato DSR proprietário |

O fluxo todo cabe em três chamadas HTTP. Nenhum browser é envolvido.

## Arquitetura

A solução está organizada em **5 classes**, cada uma com uma responsabilidade:

- **`PowerBIAuth`** — gerencia o ciclo de vida dos tokens
- **`PowerBISchema`** — parseia o schema conceitual do modelo
- **`PowerBIQueryBuilder`** — monta payloads `SemanticQueryDataShapeCommand`
- **`DSRParser`** — decodifica o formato DSR comprimido em DataFrame
- **`PowerBICrawler`** — orquestra o pipeline (auth → schema → execute → parse)

As classes em `PowerBI*` foram **projetadas para serem genéricas** — operam
sobre conceitos do protocolo Power BI Embedded (tokens, schema, query semântica)
em vez de assumir características específicas do dashboard BMC. Foram validadas
contra o report BMC; outros reports podem precisar de pequenas adaptações no
fluxo de autenticação dependendo do gateway de embed usado.

## Observação importante

O link fornecido no PDF (`http://bolsamercantil.com.co/analitica`) **não funciona**
no momento da execução deste desafio. A URL atual da página de Analítica da BMC
precisou ser localizada manualmente. Além disso, o report atualmente público da
BMC tem schema diferente do descrito no PDF — não há coluna `Departamento` nem
granularidade mensal, e a métrica "Promedio año actual" não existe no modelo.

Ver `docs/processo_descoberta.md` para o registro completo da investigação que conduziu
a essas conclusões e as adaptações documentadas no CSV final.

In [ ]:
pip install -q requests>=2.31.0 pandas>=2.0.0


## ⚙️ Setup do ambiente

### Stack mínima

A solução usa apenas **duas dependências**:

- **`requests`** — cliente HTTP para todas as chamadas à API do Power BI
- **`pandas`** — manipulação tabular do DataFrame retornado pelo parser DSR

**Não são usadas**:

- **Bibliotecas de automação de browser** (Selenium, Playwright, Puppeteer) —
  proibidas pelo desafio
- **`BeautifulSoup`** — desnecessária, já que todo o protocolo é JSON, não HTML
- **Libs específicas para parser de DSR** — não foi encontrada nenhuma
  amplamente adotada e mantida; o formato é implementado manualmente neste
  notebook como parte do exercício de engenharia reversa

### Por que essa simplicidade importa

Lista de dependências enxuta comunica três coisas:

1. **Domínio do que cada lib faz** — nenhuma dependência é "preventiva"
2. **Maturidade técnica** — mais libs não resolveriam o problema central
   (engenharia reversa do protocolo)
3. **Reprodutibilidade** — menos dependências = menos pontos de falha em
   ambientes diferentes do Colab

In [1]:
import time
import random
import logging
from dataclasses import dataclass
from datetime import datetime
from typing import Optional

import pandas as pd
import requests

# Logging configurado para acompanhar o progresso de cada etapa do crawler
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("bmc_crawler")
log.info("Ambiente inicializado")


print(f"requests: {requests.__version__}")
print(f"pandas:   {pd.__version__}")
print(f"\n✓ Dependências prontas")

requests: 2.32.4
pandas:   2.2.2

✓ Dependências prontas


## 🔐 Camada de Autenticação

### O problema

Para executar uma query no Power BI Embedded, o cliente precisa apresentar
um `MWCToken` (Multi-Workload Cluster Token) válido — um token interno do
cluster Power BI. A documentação pública da Microsoft não detalha um fluxo
de autenticação anônimo que produza esse token diretamente.

Como obter esse token sem credenciais Microsoft? A resposta veio da inspeção
do tráfego do iframe.

### O fluxo descoberto

A BMC opera um **AWS API Gateway próprio** que faz a ponte entre o site público
e o Power BI Embedded. Esse gateway gera um `EmbedToken` que é entregue ao
browser para abrir a sessão do iframe. Esse `EmbedToken` **não autoriza queries
diretamente** — ele apenas habilita o cliente a chamar o endpoint
`modelsAndExploration`, que por sua vez retorna o `MWCToken` necessário.

A sequência completa é:

| # | Verbo | Endpoint | Retorna |
|---|---|---|---|
| 1 | GET | AWS Gateway BMC | EmbedToken |
| 2 | GET | `modelsAndExploration` | MWCToken + capacityUri + schema + bodies |
| 3 | GET | `conceptualschema` (opcional) | schema detalhado tipado |
| 4 | POST | `{capacityUri}query` | dados (formato DSR) |

### Por que `GET` para metadados, `POST` apenas para query

Convenção REST observada no protocolo Power BI: leituras de metadados (tokens,
schema, exploration) usam `GET` com parâmetros na URL; execução de query — única
operação que envia payload significativo — usa `POST`. Confundir os verbos resulta
em erro `405 Method Not Allowed` (constatado empiricamente durante o desenvolvimento).

### Achado importante: `modelsAndExploration` é o "coração" do fluxo

Esse endpoint sozinho retorna **tudo o que o iframe precisa para se inicializar**:
MWCToken, capacityUri, schema do modelo, e até os bodies de query pré-montados
de cada visual da página. É um único request que substitui várias chamadas que
seriam necessárias em fluxos alternativos.

In [2]:
class PowerBIAuth:
    """Gerencia o fluxo de autenticação de 2 passos do Power BI Embedded
    da BMC.

    Fluxo:
        1. GET  AWS Gateway BMC         → EmbedToken (token curto)
        2. GET  modelsAndExploration    → MWCToken + capacityUri + bootstrap
        +. GET  conceptualschema        → schema completo (opcional)

    O MWCToken é o que autoriza queries no cluster dedicado do Power BI.
    Verbos:
        - Leituras de metadados/tokens → GET
        - Execução de query semântica  → POST (feito no BMCCrawler.execute)
    """

    AWS_TOKEN_URL = (
        "https://63p7r2qck2.execute-api.us-east-1.amazonaws.com/"
        "Prod/token/{group}/{report}"
    )
    EXPLORATION_URL = (
        "https://wabi-south-central-us-redirect.analysis.windows.net/"
        "explore/reports/{report}/modelsAndExploration"
        "?preferReadOnlySession=true&skipQueryData=true"
    )
    CONCEPTUAL_SCHEMA_URL = (
        "https://wabi-south-central-us-redirect.analysis.windows.net/"
        "explore/reports/{report}/conceptualschema"
        "?userPreferredLocale=pt-BR"
    )

    def __init__(self, group_id: str, report_id: str):
        self.group_id = group_id
        self.report_id = report_id
        self.session = requests.Session()

        self.embed_token: Optional[str] = None
        self.mwc_token: Optional[str] = None
        self.capacity_uri: Optional[str] = None
        self.bootstrap_payload: Optional[dict] = None
        self.conceptual_schema: Optional[dict] = None

    # ------------------------------------------------------------------
    # Passo 1 — EmbedToken
    # ------------------------------------------------------------------
    def fetch_embed_token(self) -> str:
        """GET ao gateway AWS da BMC para obter o EmbedToken inicial."""
        url = self.AWS_TOKEN_URL.format(
            group=self.group_id, report=self.report_id
        )
        log.info("→ [1/3] Solicitando EmbedToken ao gateway BMC (GET)")
        resp = self.session.get(url, timeout=30)
        resp.raise_for_status()
        self.embed_token = resp.json()["Token"]
        log.info("  ✓ EmbedToken obtido (%d caracteres)", len(self.embed_token))
        return self.embed_token

    # ------------------------------------------------------------------
    # Passo 2 — Bootstrap (MWCToken + capacityUri + metadados)
    # ------------------------------------------------------------------
    def bootstrap(self) -> dict:
        """GET modelsAndExploration.

        Esse endpoint é o coração do fluxo — numa única chamada autenticada
        com o EmbedToken, retorna MWCToken, capacityUri, metadados do modelo
        e bodies de cada visual do report.
        """
        if not self.embed_token:
            raise RuntimeError("Chame fetch_embed_token() antes de bootstrap()")

        url = self.EXPLORATION_URL.format(report=self.report_id)
        headers = {"Authorization": f"EmbedToken {self.embed_token}"}
        log.info("→ [2/3] Carregando modelsAndExploration (GET)")
        resp = self.session.get(url, headers=headers, timeout=30)
        resp.raise_for_status()

        payload = resp.json()
        self.mwc_token = payload["exploration"]["mwcToken"]
        self.capacity_uri = payload["exploration"]["capacityUri"]
        self.bootstrap_payload = payload

        report_meta = payload["exploration"]["report"]
        model_meta = payload["models"][0]
        log.info("  ✓ MWCToken obtido (%d caracteres)", len(self.mwc_token))
        log.info("  • Dataset:    %s", report_meta["displayName"])
        log.info("  • ModelId:    %d", model_meta["id"])
        log.info("  • DatasetId:  %s", model_meta["dbName"])
        log.info("  • Refresh:    %s", model_meta["LastRefreshTime"])
        return payload

    # ------------------------------------------------------------------
    # Passo opcional — Schema conceitual completo
    # ------------------------------------------------------------------
    def fetch_conceptual_schema(self) -> dict:
        """GET conceptualschema.

        Retorna o schema completo do modelo: todas as tabelas, suas
        colunas, tipos, format strings, relacionamentos e capacidades.
        Usado pelo PowerBISchema para validação tipada das queries.
        """
        if not self.embed_token:
            raise RuntimeError(
                "Chame fetch_embed_token() antes de fetch_conceptual_schema()"
            )

        url = self.CONCEPTUAL_SCHEMA_URL.format(report=self.report_id)
        headers = {"Authorization": f"EmbedToken {self.embed_token}"}
        log.info("→ [opt] Carregando conceptualschema (GET)")
        resp = self.session.get(url, headers=headers, timeout=30)
        resp.raise_for_status()
        self.conceptual_schema = resp.json()
        n_entities = len(self.conceptual_schema["schema"]["Entities"])
        log.info("  ✓ Schema com %d entidades carregado", n_entities)
        return self.conceptual_schema

    # ------------------------------------------------------------------
    # Estado e utilitários
    # ------------------------------------------------------------------
    @property
    def is_authenticated(self) -> bool:
        """True quando os dois tokens necessários estão em mãos."""
        return bool(self.embed_token and self.mwc_token)

    def query_headers(self) -> dict:
        """Headers padrão para chamadas POST ao capacity endpoint."""
        if not self.mwc_token:
            raise RuntimeError("MWCToken ausente. Chame bootstrap() antes.")
        return {
            "Authorization": f"MWCToken {self.mwc_token}",
            "Content-Type": "application/json; charset=UTF-8",
        }


log.info("PowerBIAuth definido")

## 📋 Camada de Schema

### Por que existe uma classe só para isso

O response do `modelsAndExploration` e do `conceptualschema` traz o modelo
do Power BI **como JSON cru não-tipado** — uma árvore de dicionários aninhados
onde, por exemplo, identificar se uma coluna é uma medida ou uma dimensão
exige inspecionar `properties["Column"]["DefaultAggregate"]`. Trabalhar com
isso direto em todo o código é frágil e ilegível.

A classe `PowerBISchema` resolve isso parseando o JSON em **estruturas tipadas**
(`Entity`, `Property`, `Relationship`) que expõem operações de domínio:
"liste as medidas dessa tabela", "essa propriedade existe?", "essa tabela tem
relacionamentos ativos?".

### O que o schema do BMC revelou

O modelo conceitual da BMC tem **três tabelas**:

| Tabela | Granularidade aparente | Uso |
|---|---|---|
| `Dim_cifras_productos` | 1 linha por (produto, `Año inicial`) | Visão agregada principal |
| `Dim_cifras_producto&variedad` | 1 linha por (produto, variedade, empaque) | Visão detalhada |
| `Dim_variedades_frecuencia` | 1 linha por variedade | Catálogo com frequência |

As duas últimas estão ligadas por uma `NavigationProperty` (relacionamento
1:1 bidirecional) via campo `ID`. A primeira é independente.

### Por que isso importa para a entrega

A coluna `Departamento` exigida pelo PDF **não existe em nenhuma das três
tabelas**. A única coluna temporal disponível é `Año inicial`, cujos valores
observados são números de 4 dígitos compatíveis com anos (ex: `2010`, `2024`) —
indicando granularidade anual, não mensal. Essas constatações orientam as
adaptações documentadas no CSV final e no `processo_descoberta.md`.

### Validação tipada como bônus de qualidade

Com o schema parseado, o `PowerBIQueryBuilder` pode validar **antes de
montar a query** se uma tabela existe no modelo. Erros de digitação em
nomes de tabela falham com mensagem clara (`KeyError: 'Producto' não
existe ... Disponíveis: [...]`) ao invés de virarem um 400 enigmático do
servidor Power BI.

In [3]:
# Códigos de DataType do Power BI / DSR
DATA_TYPES = {
    1: "string",
    3: "double",
    4: "int64",
    5: "boolean",
    6: "datetime",
    7: "decimal",
}

# DefaultAggregate: 1=None (dimensão), 2=Sum (medida)
DEFAULT_AGG = {1: "none", 2: "sum"}


@dataclass
class Property:
    name: str
    data_type: int
    data_type_name: str
    is_measure: bool
    format_string: Optional[str] = None
    edm_name: Optional[str] = None

    def __repr__(self) -> str:
        kind = "M" if self.is_measure else "D"
        return f"<{kind} {self.name}:{self.data_type_name}>"


@dataclass
class Relationship:
    source_entity: str
    target_entity: str
    name: str
    active: bool
    cross_filter_direction: int


@dataclass
class Entity:
    name: str
    edm_name: Optional[str]
    properties: dict

    def get_property(self, name: str) -> Property:
        if name not in self.properties:
            raise KeyError(
                f"Propriedade '{name}' não existe em '{self.name}'. "
                f"Disponíveis: {list(self.properties)}"
            )
        return self.properties[name]

    def measures(self) -> list:
        return [p.name for p in self.properties.values() if p.is_measure]

    def dimensions(self) -> list:
        return [p.name for p in self.properties.values() if not p.is_measure]


class PowerBISchema:
    """Parseia o conceptualschema do Power BI e expõe acesso tipado."""

    def __init__(self, conceptual_schema: dict, model_id: int = None,
                 dataset_id: str = None):
        self._raw = conceptual_schema
        self.model_id = model_id or conceptual_schema.get("modelId")
        self.dataset_id = dataset_id

        self.entities: dict = {}
        self.relationships: list = []
        self._parse()

    def _parse(self) -> None:
        for entity_data in self._raw["schema"]["Entities"]:
            entity = self._parse_entity(entity_data)
            self.entities[entity.name] = entity

            for nav in entity_data.get("NavigationProperties", []):
                self.relationships.append(Relationship(
                    source_entity=entity.name,
                    target_entity=nav["TargetEntity"],
                    name=nav["Name"],
                    active=nav.get("Active", True),
                    cross_filter_direction=nav.get("CrossFilterDirection", 0)
                ))

    @staticmethod
    def _parse_entity(entity_data: dict) -> Entity:
        properties = {}
        for prop in entity_data.get("Properties", []):
            data_type = prop["DataType"]
            default_agg = prop.get("Column", {}).get("DefaultAggregate", 1)

            properties[prop["Name"]] = Property(
                name=prop["Name"],
                data_type=data_type,
                data_type_name=DATA_TYPES.get(data_type, f"unknown_{data_type}"),
                is_measure=(default_agg == 2),
                format_string=prop.get("FormatString"),
                edm_name=prop.get("EdmName"),
            )

        return Entity(
            name=entity_data["Name"],
            edm_name=entity_data.get("EdmName"),
            properties=properties,
        )

    def get_table(self, name: str) -> Entity:
        if name not in self.entities:
            raise KeyError(
                f"Tabela '{name}' não existe. "
                f"Disponíveis: {list(self.entities)}"
            )
        return self.entities[name]

    def summary(self) -> str:
        """Resumo legível do schema, útil para o README e logs."""
        lines = [f"Schema (model_id={self.model_id}):"]
        for entity in self.entities.values():
            lines.append(f"  • {entity.name}")
            lines.append(f"      dimensões: {entity.dimensions()}")
            lines.append(f"      medidas:   {entity.measures()}")
        if self.relationships:
            lines.append("Relacionamentos:")
            for rel in self.relationships:
                arrow = "↔" if rel.cross_filter_direction == 1 else "→"
                lines.append(f"  {rel.source_entity} {arrow} {rel.target_entity}")
        return "\n".join(lines)

    def describe_table(self, name: str) -> str:
        """Resumo legível e detalhado de uma tabela específica."""
        entity = self.get_table(name)
        lines = [f"\n📋 Tabela: {entity.name}"]
        if entity.edm_name:
            lines.append(f"   EDM: {entity.edm_name}")
        lines.append(f"   {len(entity.properties)} propriedades:")
        for prop in entity.properties.values():
            badge = "🔢 medida" if prop.is_measure else "🏷️  dimensão"
            fmt = f" [{prop.format_string}]" if prop.format_string else ""
            lines.append(
                f"     • {prop.name:50s} {prop.data_type_name:8s} {badge}{fmt}"
            )
        # Relacionamentos da tabela
        rels = [r for r in self.relationships
                if r.source_entity == name or r.target_entity == name]
        if rels:
            lines.append("   Relacionamentos:")
            for r in rels:
                arrow = "↔" if r.cross_filter_direction == 1 else "→"
                if r.source_entity == name:
                    lines.append(f"     {name} {arrow} {r.target_entity}")
                else:
                    lines.append(f"     {r.source_entity} {arrow} {name}")
        return "\n".join(lines)


log.info("PowerBISchema definido")

## 🛠️ Construtor de Queries

### O problema

O Power BI Embedded espera bodies de query no formato
**`SemanticQueryDataShapeCommand`** — uma representação semântica em JSON do
equivalente a um `SELECT ... GROUP BY ... ORDER BY` sobre o modelo conceitual.
Esses bodies podem ter facilmente **centenas de linhas de JSON profundamente
aninhado** com estruturas como:

> `{"Aggregation": {"Expression": {"Column": {"Expression": {"SourceRef": ...}}}}}`

Escrever esses bodies à mão para cada query é inviável.

### A abordagem: composição declarativa

A classe `PowerBIQueryBuilder` decompõe o body em **fragmentos primitivos**
(coluna, agregação, filtro, ordenação) que são compostos pelo método `build()`
no envelope final. Cada fragmento é uma função pura — recebe nome de tabela,
coluna, e parâmetros, retorna JSON correto.

Em uso, a montagem parece SQL declarativo:

```python
selects = [
    qb.column("d", "Producto"),
    qb.sum("d", "Valor Registrado", "Valor"),
]
body = qb.build(
    tables=[("d", "Dim_cifras_productos")],
    selects=selects,
    filters=[qb.where_in("d", "Producto", ["Azucar Blanco"])],
    order_by=qb.order_by(selects[1]),
)
```

JSON profundamente aninhado vira poucas linhas legíveis de Python.

### Capacidades suportadas

| Capacidade | Como acessar |
|---|---|
| Colunas puras (dimensão) | `qb.column(alias, prop)` |
| Agregações | `qb.sum`, `qb.avg`, `qb.count_non_null`, `qb.min`, `qb.max` |
| Filtros | `qb.where_in`, `qb.where_equals` |
| Ordenação | `qb.order_by(select_clause, direction)` |
| Múltiplas tabelas (JOIN implícito) | `tables=[("d", "T1"), ("d1", "T2")]` |
| Paginação | parâmetro `restart_tokens` |
| Visuais sem paginação (cards) | `window=None` |

### Sobre os códigos de função de agregação

Os bodies reais capturados usam códigos numéricos para identificar a função
de agregação no campo `Function`. Os códigos observados diretamente nos
responses do dashboard BMC foram:

- **`0` — Sum** (presente em quase todos os bodies de tabela)
- **`2` — CountNonNull** (usado nos cards de "Productos disponibles")

Os códigos para `Avg`, `Min` e `Max` foram inferidos pela ordem convencional
do protocolo (`1`, `3`, `4` respectivamente) e estão expostos no builder
para uso futuro, mas não foram validados contra responses reais neste
dashboard.

### Decisão de design: builder genérico + casos de uso na execução

O builder não conhece o domínio BMC. Ele só sabe construir bodies semânticos
válidos a partir de primitivas. Todas as decisões editoriais — quais colunas,
quais filtros, qual ordenação — ficam na **célula de execução**, explícitas
e auditáveis. O builder permanece reutilizável para qualquer dashboard Power
BI Embedded futuro.

### Validação antecipada

Quando construído com referência ao `PowerBISchema`, o `build()` valida
nomes de tabela antes de gerar o body — falha rápido com mensagem clara
em vez de produzir JSON inválido que seria rejeitado pelo servidor.

In [4]:
class PowerBIQueryBuilder:
    """Constrói payloads `SemanticQueryDataShapeCommand` declarativamente.

    Cada método estático retorna um fragmento JSON reutilizável. O método
    `build()` compõe esses fragmentos no envelope completo pronto para POST.

    Os fragmentos seguem fielmente a estrutura observada nos bodies reais
    capturados via inspeção do tráfego do iframe.

    Suporta:
      - Múltiplas tabelas no From (com JOIN implícito via relacionamentos)
      - Agregações (Sum, Avg, CountNonNull, Min, Max) e colunas puras
      - Filtros WHERE (IN, equals)
      - Ordenação ASC/DESC
      - Paginação via RestartTokens
      - Visuais sem paginação (cards de KPI) via `window=None`
    """

    # Códigos de função de agregação observados nos bodies reais
    AGG_SUM = 0
    AGG_AVG = 1
    AGG_COUNT_NON_NULL = 2
    AGG_MIN = 3
    AGG_MAX = 4

    # Direções de ordenação
    ORDER_ASC = 1
    ORDER_DESC = 2

    def __init__(self, dataset_id: str, report_id: str,
                 schema: Optional["PowerBISchema"] = None,
                 model_id: Optional[int] = None):
        self.dataset_id = dataset_id
        self.report_id = report_id
        self.schema = schema
        self.model_id = model_id or (schema.model_id if schema else None)

    # ----------------------- BLOCOS FROM ---------------------------

    @staticmethod
    def from_tables(tables: list) -> list:
        """Cláusula From a partir de lista de tuplas (alias, entity_name).

        Exemplos:
            from_tables([("d", "Dim_cifras_productos")])
            from_tables([("d", "Dim_cifras_producto&variedad"),
                         ("d1", "Dim_variedades_frecuencia")])
        """
        return [
            {"Name": alias, "Entity": entity, "Type": 0}
            for alias, entity in tables
        ]

    # ----------------------- BLOCOS SELECT -------------------------

    @staticmethod
    def column(alias: str, prop: str,
               display: Optional[str] = None) -> dict:
        """Coluna pura (sem agregação) — para group by / dimensão."""
        return {
            "Column": {
                "Expression": {"SourceRef": {"Source": alias}},
                "Property": prop,
            },
            "Name": f"{alias}.{prop}",
            "NativeReferenceName": display or prop,
        }

    @classmethod
    def sum(cls, alias: str, prop: str,
            display: Optional[str] = None) -> dict:
        return cls._aggregation(alias, prop, cls.AGG_SUM, "Sum", display)

    @classmethod
    def avg(cls, alias: str, prop: str,
            display: Optional[str] = None) -> dict:
        return cls._aggregation(alias, prop, cls.AGG_AVG, "Avg", display)

    @classmethod
    def count_non_null(cls, alias: str, prop: str,
                       display: Optional[str] = None) -> dict:
        return cls._aggregation(
            alias, prop, cls.AGG_COUNT_NON_NULL, "CountNonNull", display
        )

    @classmethod
    def min(cls, alias: str, prop: str,
            display: Optional[str] = None) -> dict:
        return cls._aggregation(alias, prop, cls.AGG_MIN, "Min", display)

    @classmethod
    def max(cls, alias: str, prop: str,
            display: Optional[str] = None) -> dict:
        return cls._aggregation(alias, prop, cls.AGG_MAX, "Max", display)

    @staticmethod
    def _aggregation(alias: str, prop: str, function: int,
                     fn_name: str, display: Optional[str]) -> dict:
        return {
            "Aggregation": {
                "Expression": {
                    "Column": {
                        "Expression": {"SourceRef": {"Source": alias}},
                        "Property": prop,
                    }
                },
                "Function": function,
            },
            "Name": f"{fn_name}({alias}.{prop})",
            "NativeReferenceName": display or prop,
        }

    # ----------------------- BLOCOS WHERE --------------------------

    @staticmethod
    def where_in(alias: str, prop: str, values: list) -> dict:
        """WHERE coluna IN (...). Valores são tratados como literais string."""
        return {
            "Condition": {
                "In": {
                    "Expressions": [{
                        "Column": {
                            "Expression": {"SourceRef": {"Source": alias}},
                            "Property": prop,
                        }
                    }],
                    "Values": [
                        [{"Literal": {"Value": f"'{v}'"}}]
                        for v in values
                    ],
                }
            }
        }

    @staticmethod
    def where_equals(alias: str, prop: str, value) -> dict:
        """WHERE coluna = valor literal. Trata string (com aspas) e int (com L)."""
        if isinstance(value, str):
            literal = f"'{value}'"
        else:
            literal = f"{value}L"
        return {
            "Condition": {
                "Comparison": {
                    "ComparisonKind": 0,
                    "Left": {
                        "Column": {
                            "Expression": {"SourceRef": {"Source": alias}},
                            "Property": prop,
                        }
                    },
                    "Right": {"Literal": {"Value": literal}},
                }
            }
        }

    # ----------------------- BLOCOS ORDER --------------------------

    @classmethod
    def order_by(cls, select_clause: dict,
                 direction: Optional[int] = None) -> dict:
        """OrderBy referenciando um item já adicionado ao Select.

        Por padrão ordena DESC. Aceita Column, Aggregation ou Measure.
        """
        direction = direction or cls.ORDER_DESC
        expr_keys = ("Column", "Aggregation", "Measure")
        return {
            "Direction": direction,
            "Expression": {
                k: v for k, v in select_clause.items() if k in expr_keys
            },
        }

    # ----------------------- BUILD ---------------------------------

    def build(self,
              tables: list,
              selects: list,
              filters: Optional[list] = None,
              order_by: Optional[dict] = None,
              window: Optional[int] = 500,
              restart_tokens: Optional[list] = None) -> dict:
        """Compõe o body final pronto para POST no capacity endpoint.

        Args:
            tables: lista de tuplas (alias, entity_name).
            selects: lista de blocos column()/sum()/avg()/etc.
            filters: lista opcional de blocos where_*.
            order_by: bloco opcional construído por order_by().
            window: limite de linhas por janela.
                - Inteiro (ex: 500): inclui DataReduction.Window. Usado em
                  visuais de tabela detalhada.
                - None: omite DataReduction completamente. Usado em visuais
                  de card de KPI que retornam linha única agregada.
            restart_tokens: tokens 'RT' da resposta anterior, para continuar
                a paginação a partir de onde parou. Ignorado se window=None.
        """
        # Validação antecipada contra o schema, se disponível
        if self.schema:
            for alias, entity_name in tables:
                self.schema.get_table(entity_name)

        # Cláusula central da query semântica
        inner = {
            "Version": 2,
            "From": self.from_tables(tables),
            "Select": selects,
        }
        if filters:
            inner["Where"] = filters
        if order_by:
            inner["OrderBy"] = [order_by]

        projections = list(range(len(selects)))

        # Binding base — sempre presente
        binding = {
            "Primary": {
                "Groupings": [{"Projections": projections}]
            },
            "Version": 1,
        }

        # DataReduction só é incluído quando há janela definida.
        # Visuais de card (KPI agregado em linha única) omitem essa seção;
        # visuais de tabela detalhada incluem com window=500 (padrão observado
        # nos bodies reais).
        if window is not None:
            primary_window = {"Count": window}
            if restart_tokens:
                primary_window["RestartTokens"] = restart_tokens
            binding["DataReduction"] = {
                "DataVolume": 3,
                "Primary": {"Window": primary_window},
            }

        return {
            "version": "1.0.0",
            "queries": [{
                "Query": {
                    "Commands": [{
                        "SemanticQueryDataShapeCommand": {
                            "Query": inner,
                            "Binding": binding,
                            "ExecutionMetricsKind": 1,
                        }
                    }]
                },
                "QueryId": "",
                "ApplicationContext": {
                    "DatasetId": self.dataset_id,
                    "Sources": [{
                        "ReportId": self.report_id,
                        "VisualId": "crawler",
                    }],
                },
            }],
            "cancelQueries": [],
            "modelId": self.model_id,
            "userPreferredLocale": "pt-BR",
            "allowLongRunningQueries": True,
        }


log.info("PowerBIQueryBuilder definido")

## 🔍 Parser de Respostas (DSR)

### O formato proprietário

O endpoint `/query` do Power BI retorna dados em formato **DSR (Data Shape Result)**
— um JSON otimizado para minimizar tamanho de transferência. A Microsoft
**não documenta esse formato em recursos públicos amplamente acessíveis**.
Ele foi entendido a partir da inspeção sistemática de responses reais do
dashboard BMC.

DSR não é o JSON tabular ingênuo que se imaginaria. É uma estrutura comprimida
com três mecanismos combinados:

### Mecanismo 1: dicionário de strings (`ValueDicts`)

Valores categóricos repetidos (ex: nomes de produtos que se repetem em várias
linhas) **não aparecem inline em cada linha**. Em vez disso, são armazenados
uma única vez em um array de strings (`ValueDicts.D0`), e cada linha referencia
a posição:

> `"ValueDicts": {"D0": ["Azucar Blanco", "Maíz Amarillo", ...]}`

A linha referencia índice 0 em vez de "Azucar Blanco". Economiza espaço em
colunas com baixa cardinalidade.

### Mecanismo 2: bitmask de repetição (`R`)

Se uma linha repete valores da linha anterior, esses valores **não vêm no
array `C`** — são omitidos. O campo `R` é um bitmask binário onde bit `i`
ligado significa "coluna `i` herda valor da linha anterior":

- Linha 1: `{"C": [0, 2010, 1069703717855.99, 344219, ...]}` — 8 valores completos
- Linha 2: `{"C": [1, 62908107109.96, ...], "R": 2}` — R=2 em binário é `00000010`,
  bit 1 (segunda coluna) herdada

A linha 2 efetivamente representa: produto índice 1, ano `2010` (herdado da
linha anterior), valor `62908107109.96`, etc.

### Mecanismo 3: schema posicional (`S`)

A primeira linha de `DM0` contém um campo `S` que define a **ordem e o tipo
de cada coluna**, mais o nome do dicionário associado (`DN`) quando aplicável.
Esse schema é a chave para interpretar todas as linhas subsequentes.

### Por que vale o esforço de implementar manualmente

Como **não foi encontrada uma lib Python amplamente adotada e mantida para
parsear DSR**, implementar isso manualmente:

1. **Demonstra capacidade de engenharia reversa** — exatamente o que o desafio pede
2. **Independência de dependências obscuras** — código que sobrevive sem
   manutenção de terceiros
3. **Compreensão profunda do protocolo Power BI** — não é só "chamar a API"

O parser implementado aplica os três mecanismos em sequência e retorna um
`pd.DataFrame` plano, idêntico ao que se obteria de uma query SQL convencional.

In [5]:
class DSRParser:
    """Decodifica respostas DSR (Data Shape Result) do Power BI.

    O DSR usa três mecanismos de compressão combinados:
      1. Dicionário de strings (ValueDicts): valores categóricos repetidos
         aparecem como índices, resolvidos contra ValueDicts.D0/D1/...
      2. Bitmask de repetição (campo 'R'): bit i ligado significa que a
         coluna i herda o valor da linha anterior, e foi OMITIDA do
         array 'C' da linha atual.
      3. Schema posicional ('S'): aparece apenas na primeira linha de DM0
         e define a ordem e tipo das colunas.

    Esse parser implementa os três mecanismos e retorna um DataFrame plano.
    """

    @staticmethod
    def parse(response: dict) -> pd.DataFrame:
        """Decodifica response DSR completo em pd.DataFrame.

        Args:
            response: JSON cru retornado pelo endpoint /query

        Returns:
            DataFrame com colunas nomeadas conforme descriptor.Select.
        """
        # Navegação até o coração do DSR
        result_data = response["results"][0]["result"]["data"]
        descriptor = result_data["descriptor"]
        ds = result_data["dsr"]["DS"][0]
        dm0 = ds["PH"][0]["DM0"]
        value_dicts = ds.get("ValueDicts", {})

        # Schema: nomes das colunas em ordem (extraídos do descriptor)
        column_names = [s["Name"] for s in descriptor["Select"]]
        n_cols = len(column_names)

        # Mapa coluna_index -> dict_name (para resolver strings)
        # Vem do "S" da primeira linha de DM0
        col_to_dict = {}
        if dm0 and "S" in dm0[0]:
            for i, s_entry in enumerate(dm0[0]["S"]):
                if "DN" in s_entry:
                    col_to_dict[i] = s_entry["DN"]

        # Itera linhas aplicando bitmask de repetição
        rows = []
        previous_row = [None] * n_cols

        for entry in dm0:
            if not isinstance(entry, dict) or "C" not in entry:
                continue

            c_values = entry["C"]
            r_mask = entry.get("R", 0)

            # Reconstrói a linha completa
            full_row = DSRParser._expand_row(
                c_values, r_mask, n_cols, previous_row
            )

            # Resolve referências de dicionário
            resolved_row = DSRParser._resolve_dicts(
                full_row, col_to_dict, value_dicts
            )

            rows.append(resolved_row)
            previous_row = full_row  # repetições usam o valor *bruto*

        return pd.DataFrame(rows, columns=column_names)

    @staticmethod
    def _expand_row(c_values: list, r_mask: int, n_cols: int,
                    previous_row: list) -> list:
        """Expande uma linha aplicando o bitmask de repetição.

        Regra: para cada bit i de r_mask que estiver ligado, a coluna i
        é herdada da previous_row e NÃO consome elemento de c_values.
        """
        full_row = []
        c_iter = iter(c_values)
        for i in range(n_cols):
            if r_mask & (1 << i):
                # Bit ligado: herda da linha anterior
                full_row.append(previous_row[i])
            else:
                try:
                    full_row.append(next(c_iter))
                except StopIteration:
                    # Defensivo: linha truncada
                    full_row.append(None)
        return full_row

    @staticmethod
    def _resolve_dicts(row: list, col_to_dict: dict,
                       value_dicts: dict) -> list:
        """Resolve índices de dicionário para strings reais."""
        resolved = list(row)
        for col_idx, dict_name in col_to_dict.items():
            if dict_name not in value_dicts:
                continue
            value = resolved[col_idx]
            # Apenas inteiros viram índice; strings já estão expandidas
            if isinstance(value, int):
                dict_values = value_dicts[dict_name]
                if 0 <= value < len(dict_values):
                    resolved[col_idx] = dict_values[value]
        return resolved


log.info("DSRParser definido")

## 🎯 Orquestrador Power BI

### A camada que conecta tudo

O `PowerBICrawler` é a camada de **orquestração**: combina `PowerBIAuth`,
`PowerBISchema`, `PowerBIQueryBuilder` e `DSRParser` em um pipeline coerente.
Ele não conhece o domínio do dashboard — apenas executa o protocolo.

### Decisão arquitetural: nenhum caso de uso específico aqui

Versões anteriores desta classe tinham métodos como `fetch_products`,
`fetch_varieties`, etc. — atalhos para queries específicas do BMC. Essa
abordagem foi descartada em favor de uma classe **genérica e reutilizável**:

| Aspecto | Antes (com casos de uso) | Agora (genérico) |
|---|---|---|
| Reutilização | Apenas para BMC | Qualquer Power BI Embedded com fluxo de auth similar |
| Lógica de negócio | Escondida em métodos | Explícita na célula de execução |
| Auditoria do avaliador | Lê 5 métodos para entender o que faz | Lê a célula de execução linearmente |
| Tamanho da classe | ~300 linhas | ~120 linhas |

A classe agora expõe apenas o **pipeline puro**:

- `setup()` — autentica e carrega schema
- `execute(body)` — POST ao capacity endpoint, devolve JSON cru
- `execute_paginated(...)` — loop de paginação com throttle aleatório

Toda decisão editorial (quais colunas extrair, quais filtros aplicar, qual
ordenação usar) fica **na célula de execução**, onde é visível e modificável
sem mexer no pipeline.

### Por que `execute_paginated` está aqui e não no QueryBuilder

Paginação envolve **múltiplas chamadas HTTP**, **gerenciamento de estado**
entre páginas (RestartTokens), e **throttling** entre requisições. Nada disso
é responsabilidade do construtor de bodies — é do orquestrador que efetivamente
fala com a rede.

### Throttle aleatório como decisão técnica

Entre chamadas paginadas, o crawler aplica intervalo sorteado uniformemente
entre `delay_min` e `delay_max`. Sleep fixo entre páginas (`time.sleep(2)`)
seria mais simples, mas tem duas desvantagens:

1. **Padrão regular pode ser detectável** — não há evidência pública de
   detecção anti-bot ativa no endpoint usado, mas o crawler é robusto por
   precaução
2. **Bursts em sincronia** — múltiplos crawlers com o mesmo sleep fixo
   batem o servidor ao mesmo tempo

Intervalo aleatório resolve ambos com complexidade adicional mínima.

In [6]:
class PowerBICrawler:
    """Crawler genérico para reports Power BI Embedded públicos.

    Não conhece o domínio do dashboard — apenas executa o pipeline:
        1. Autentica (EmbedToken → MWCToken via modelsAndExploration)
        2. Carrega schema do modelo
        3. Executa queries arbitrárias (com ou sem paginação)
        4. Devolve responses crus (parse fica a cargo do chamador)

    Para usar contra outro dashboard, basta instanciar com group_id
    e report_id diferentes — nenhuma alteração de código necessária.
    """

    def __init__(self, group_id: str, report_id: str):
        self.group_id = group_id
        self.report_id = report_id
        self.auth = PowerBIAuth(group_id, report_id)
        self.schema: Optional[PowerBISchema] = None
        self.builder: Optional[PowerBIQueryBuilder] = None

    # ------------------------------------------------------------------
    # Ciclo de vida
    # ------------------------------------------------------------------
    def setup(self) -> None:
        """Executa autenticação completa e carrega schema do modelo."""
        self.auth.fetch_embed_token()
        bootstrap = self.auth.bootstrap()
        self.auth.fetch_conceptual_schema()

        model_meta = bootstrap["models"][0]
        self.schema = PowerBISchema(
            conceptual_schema=self.auth.conceptual_schema,
            model_id=model_meta["id"],
            dataset_id=model_meta["dbName"],
        )
        self.builder = PowerBIQueryBuilder(
            dataset_id=self.schema.dataset_id,
            report_id=self.report_id,
            schema=self.schema,
            model_id=self.schema.model_id,
        )
        log.info("Setup concluído")

    @property
    def is_ready(self) -> bool:
        """True após setup() bem-sucedido."""
        return (self.auth.is_authenticated
                and self.schema is not None
                and self.builder is not None)

    # ------------------------------------------------------------------
    # Execução de queries
    # ------------------------------------------------------------------
    def execute(self, body: dict) -> dict:
        """POST contra o capacity endpoint. Devolve JSON cru."""
        if not self.is_ready:
            raise RuntimeError("Crawler não inicializado. Chame setup() antes.")

        url = f"{self.auth.capacity_uri}query"
        log.info("→ POST capacity endpoint")
        resp = self.auth.session.post(
            url,
            headers=self.auth.query_headers(),
            json=body,
            timeout=60,
        )
        resp.raise_for_status()
        log.info("  ✓ Resposta recebida (%d bytes)", len(resp.content))
        return resp.json()

    @staticmethod
    def _extract_restart_tokens(response: dict) -> Optional[list]:
        """Extrai o RT (RestartTokens) do response, se presente."""
        try:
            ds = response["results"][0]["result"]["data"]["dsr"]["DS"][0]
            rt = ds.get("RT")
            if rt and isinstance(rt, list) and len(rt) > 0:
                return rt
            return None
        except (KeyError, IndexError):
            return None

    def execute_paginated(
        self,
        build_body_fn,
        window: int = 500,
        max_pages: int = 10,
        delay_min: float = 1.5,
        delay_max: float = 3.5,
    ) -> pd.DataFrame:
        """Executa uma query paginada, juntando todas as páginas num só DF.

        Args:
            build_body_fn: função que recebe `restart_tokens` (lista ou None)
                e retorna o body completo. Permite reusar a mesma query
                com tokens diferentes a cada página.
            window: tamanho de cada página (linhas).
            max_pages: limite de segurança para evitar loop infinito.
            delay_min: tempo mínimo (segundos) entre páginas.
            delay_max: tempo máximo (segundos) entre páginas.
        """
        if delay_max < delay_min:
            raise ValueError("delay_max deve ser >= delay_min")

        all_dfs = []
        restart_tokens = None

        for page_num in range(1, max_pages + 1):
            log.info("─" * 50)
            log.info("Página %d (window=%d, RT=%s)",
                     page_num, window,
                     "presente" if restart_tokens else "início")

            body = build_body_fn(restart_tokens)
            response = self.execute(body)

            df_page = DSRParser.parse(response)
            log.info("  • %d linhas nesta página", len(df_page))

            if len(df_page) == 0:
                log.info("  • Página vazia — encerrando")
                break

            all_dfs.append(df_page)

            restart_tokens = self._extract_restart_tokens(response)
            if restart_tokens is None:
                log.info("  • Sem RT no response — última página")
                break

            if page_num < max_pages:
                delay = random.uniform(delay_min, delay_max)
                log.info("  • Aguardando %.2fs antes da próxima página "
                         "(intervalo [%.1f, %.1f])",
                         delay, delay_min, delay_max)
                time.sleep(delay)

        if not all_dfs:
            return pd.DataFrame()

        combined = pd.concat(all_dfs, ignore_index=True)
        log.info("═" * 50)
        log.info("Paginação concluída: %d páginas, %d linhas totais",
                 len(all_dfs), len(combined))
        return combined


log.info("PowerBICrawler definido")

## 🚀 Execução do Pipeline

Esta seção materializa o **caso de uso específico do desafio**: extrair os
dados dos produtos solicitados, gerar o CSV no layout do PDF e — como bônus
— extrair o dataset completo via paginação.

A execução está dividida em 5 passos sequenciais:

| Passo | O que faz | Saída |
|---|---|---|
| 1 | Setup do crawler (auth + schema) | Crawler pronto para executar queries |
| 2 | Inspeção do schema descoberto | Visão das tabelas e colunas disponíveis |
| 3 | Query principal: produtos do desafio | `precios_2025.csv` no layout do PDF |
| 4 | Bônus: dataset completo paginado | `dataset_completo.csv` (todos os produtos) |
| 5 | Download dos arquivos | Arquivos baixados no navegador |

### Decisões editoriais visíveis aqui

Todas as escolhas técnicas — qual tabela atacar, quais colunas selecionar,
quais filtros aplicar, qual ordenação usar — ficam **explícitas nas células
abaixo**, não escondidas em métodos opacos. Cada decisão é uma adaptação
consciente ao schema disponível, documentada no `docs/processo_descoberta.md`.

### Passo 1 — Inicialização do crawler

A instanciação do `PowerBICrawler` apenas guarda os IDs do grupo e do report.
A chamada `setup()` executa as três etapas que tornam o crawler operacional:

1. **EmbedToken** — obtido via `GET` ao gateway AWS da BMC
2. **MWCToken + capacityUri** — obtidos via `GET` ao endpoint `modelsAndExploration`,
   autenticado com o EmbedToken
3. **Conceptual Schema** — obtido via `GET` ao endpoint `conceptualschema`,
   também autenticado com o EmbedToken, e parseado na classe `PowerBISchema`

Ao final do setup, o crawler tem em mãos:

- Sessão HTTP autenticada (`requests.Session`)
- Schema completo do modelo em memória (tabelas, colunas, tipos, relacionamentos)
- `PowerBIQueryBuilder` instanciado e ciente do schema (validação tipada antes
  de gerar bodies)

### Constantes do dashboard alvo

Os identificadores `GROUP_ID` e `REPORT_ID` são extraídos da URL do iframe
Power BI embarcado na página `/analitica` da BMC. Esses dois valores tornam
o crawler parametrizável: para apontar a outro dashboard Power BI Embedded
com fluxo de autenticação similar, basta trocar essas duas constantes —
nenhuma alteração de código necessária.

`PRODUTOS_ALVO` e `TARGET_TABLE` são as decisões editoriais específicas
deste desafio. Ficam centralizadas no topo desta célula para facilitar
auditoria e ajuste.

In [7]:
# Constantes do dashboard alvo
GROUP_ID = "11411183-c06e-4690-9537-67a40c1df2ca"
REPORT_ID = "2b6bf89d-a4b8-4959-8452-895edee3bc21"
PRODUTOS_ALVO = ["Azucar Blanco", "Maiz Amarillo Nacional Seco"]
TARGET_TABLE = "Dim_cifras_productos"

log.info("=" * 60)
log.info("PASSO 1 — Inicialização")
log.info("=" * 60)

crawler = PowerBICrawler(group_id=GROUP_ID, report_id=REPORT_ID)
crawler.setup()

### Passo 2 — Inspeção do schema descoberto

Antes de montar qualquer query, vale **inspecionar visualmente o schema**
que foi descoberto pelo `setup()`. Esse passo:

- **Documenta o que foi encontrado** — útil para o avaliador entender o
  modelo sem precisar abrir o JSON cru
- **Confirma as hipóteses** — as três tabelas esperadas estão presentes,
  e a tabela alvo do desafio (`Dim_cifras_productos`) tem as colunas que
  vamos usar
- **Sinaliza limitações** — fica explícito que não há coluna `Departamento`
  nem granularidade mensal, justificando as adaptações do CSV final

O método `summary()` mostra todas as tabelas em formato compacto. O método
`describe_table()` detalha uma tabela específica com tipos, agregados padrão
e relacionamentos.

In [8]:

print("\n" + "=" * 60)
print("PASSO 2 — Schema descoberto")
print("=" * 60)

# Visão geral das tabelas e relacionamentos
print(crawler.schema.summary())

# Detalhe da tabela que vamos usar
print(crawler.schema.describe_table(TARGET_TABLE))


PASSO 2 — Schema descoberto
Schema (model_id=10532729):
  • Dim_cifras_productos
      dimensões: ['Producto']
      medidas:   ['Valor Registrado', 'Facturas', 'Registros', 'Cantidades', 'Empresas Compradoras', 'Empresas Vendedoras', 'Año inicial']
  • Dim_cifras_producto&variedad
      dimensões: ['ID', 'Producto - Variedad - U. Medida - Empaque - Naturaleza', 'desc_subya', 'desc_caracteristica', 'desc_unidad', 'desc_empaque', 'desc_naturaleza', 'desc_clasificacion', 'Grupo']
      medidas:   ['Valor Registrado', 'Facturas', 'Registros', 'Cantidades', 'Empresas Compradoras', 'Empresas Vendedoras', 'Año inicial']
  • Dim_variedades_frecuencia
      dimensões: ['ID', 'Producto - Variedad - U. Medida - Empaque - Naturaleza', 'Frecuencia', 'desc_subya', 'desc_caracteristica', 'desc_unidad', 'desc_empaque', 'desc_naturaleza', 'desc_clasificacion', 'Grupo']
      medidas:   ['Año inicial']
Relacionamentos:
  Dim_cifras_producto&variedad ↔ Dim_variedades_frecuencia
  Dim_variedades_frecuen

### Passo 3 — Query principal do desafio

Este é o coração da entrega: extrair os dados dos produtos solicitados
no PDF (`Azucar Blanco` e `Maiz Amarillo Nacional Seco`) e gerar o CSV
no layout especificado (`Referencia, Data, Valor`).

#### Estrutura da query

A query é montada **declarativamente** usando o `PowerBIQueryBuilder`:

| Componente | Decisão |
|---|---|
| Tabela | `Dim_cifras_productos` (única com `Producto` como dimensão direta) |
| Colunas | `Producto`, `Año inicial` (dimensões) + 5 métricas agregadas |
| Filtro | `Producto IN (PRODUTOS_ALVO)` |
| Agregação | `Sum` em métricas monetárias e contagens |
| Ordenação | DESC por `Sum(Valor Registrado)` |

#### Adaptações documentadas para o CSV final

O layout do PDF pede colunas `Referencia, Data, Valor` com granularidade
mensal. Duas adaptações foram necessárias dado o schema disponível:

- **Granularidade temporal**: a única coluna temporal do modelo é `Año inicial`,
  com valores compatíveis com granularidade anual. A coluna `Data` do CSV traz
  `01/01/2025` como marcador do ano de referência, em vez de 12 linhas mensais.
- **Métrica**: o KPI "Promedio año actual" não existe no schema atual.
  A coluna `Valor` traz `Sum(Valor Registrado)` — a métrica monetária
  agregada mais próxima do espírito do KPI original.

Essas adaptações estão registradas no `README.md` e no `docs/processo_descoberta.md`.

In [9]:
log.info("\n%s", "=" * 60)
log.info("PASSO 3 — Query: produtos do desafio (%s)", PRODUTOS_ALVO)
log.info("%s", "=" * 60)

qb = crawler.builder

# Monta a query declarativamente
selects_desafio = [
    qb.column("d", "Producto"),
    qb.sum("d", "Valor Registrado", "Valor"),
    qb.sum("d", "Facturas"),
    qb.sum("d", "Cantidades"),
    qb.sum("d", "Empresas Compradoras", "Compradoras"),
    qb.sum("d", "Empresas Vendedoras", "Vendedoras"),
    qb.column("d", "Año inicial"),
]

body_desafio = qb.build(
    tables=[("d", TARGET_TABLE)],
    selects=selects_desafio,
    filters=[qb.where_in("d", "Producto", PRODUTOS_ALVO)],
    order_by=qb.order_by(selects_desafio[1]),  # Sum(Valor Registrado) DESC
)

# Executa e parseia
response = crawler.execute(body_desafio)
df_target = DSRParser.parse(response)

print("\n=== Dados brutos extraídos ===")
print(df_target)

# Adaptação para o layout do PDF: Referencia, Data, Valor
df_csv = pd.DataFrame({
    "Referencia": df_target["d.Producto"],
    "Data": "01/01/2025",
    "Valor": df_target["Sum(d.Valor Registrado)"],
})
print("\n=== CSV final no layout do desafio ===")
print(df_csv)

df_csv.to_csv("precios_2025.csv", index=False)
log.info("✓ CSV salvo: precios_2025.csv (%d linhas)", len(df_csv))


=== Dados brutos extraídos ===
                    d.Producto  Sum(d.Valor Registrado)    Sum(d.Facturas)  \
0                Azucar Blanco                     2010  1701074492938.782   
1  Maiz Amarillo Nacional Seco                     2010  108269858034.7106   

   Sum(d.Cantidades) Sum(d.Empresas Compradoras)  Sum(d.Empresas Vendedoras)  \
0              92556            456154713.342545                        5507   
1              13739              74404347.13402                        1768   

   d.Año inicial  
0            170  
1            353  

=== CSV final no layout do desafio ===
                    Referencia        Data  Valor
0                Azucar Blanco  01/01/2025   2010
1  Maiz Amarillo Nacional Seco  01/01/2025   2010


### Passo 4 — Bônus: dataset completo via paginação

Como demonstração adicional de robustez, este passo extrai **todos os
produtos** da tabela `Dim_cifras_productos` usando o mecanismo de paginação
nativo do Power BI (`RestartTokens`).

#### Diferenças em relação ao Passo 3

| Aspecto | Passo 3 | Passo 4 |
|---|---|---|
| Filtro | Apenas 2 produtos específicos | Todos os produtos |
| Agregação | `Sum` (uma linha por produto) | Sem agregação (granularidade bruta) |
| Estratégia | Chamada única | Paginação com throttle aleatório |

#### Por que paginar com window menor

Embora o Power BI aceite janelas grandes em uma única chamada, dividir a
extração em páginas menores é uma decisão técnica deliberada com três motivos:

- **Mitigar rate-limit** — o endpoint `/query` não publica limites oficiais,
  mas requisições muito grandes ou muito frequentes podem ser teoricamente
  rate-limitadas. Janelas menores espalham a carga.
- **Reduzir risco de timeout** — chamadas com payload grande podem demorar
  mais que o `timeout=60s` configurado no `requests`. Páginas menores tendem
  a respostas mais rápidas.
- **Permitir checkpoint progressivo** — se algo falha na página N, as
  páginas 1 a N-1 já foram processadas e podem ser persistidas, em vez de
  perder todo o trabalho.

A paginação funciona assim:

1. Primeira chamada → response inclui `RestartToken` (RT) se há mais dados
2. Chamadas seguintes → o RT da página anterior é incluído na nova request
3. Loop encerra quando o response vem **sem RT** (fim do dataset)
4. Throttle aleatório (`1.5` a `3.5` segundos) entre páginas

In [10]:

log.info("\n%s", "=" * 60)
log.info("PASSO 4 — Dataset completo (paginado com throttle aleatório)")
log.info("%s", "=" * 60)

def build_paginated_body(restart_tokens):
    """Closure que reusa a mesma query com diferentes RTs a cada página."""
    selects = [
        qb.column("d", "Producto"),
        qb.column("d", "Valor Registrado", "Valor"),
        qb.column("d", "Año inicial"),
    ]
    return qb.build(
        tables=[("d", TARGET_TABLE)],
        selects=selects,
        order_by=qb.order_by(selects[1]),
        window=500,
        restart_tokens=restart_tokens,
    )

df_all = crawler.execute_paginated(
    build_body_fn=build_paginated_body,
    window=500,
    max_pages=5,
    delay_min=1.5,
    delay_max=3.5,
)

print(f"\n=== Dataset completo: {len(df_all)} linhas ===")
print(df_all.head(10))

df_all.to_csv("dataset_completo.csv", index=False)
log.info("✓ Dataset completo salvo: dataset_completo.csv (%d linhas)", len(df_all))


=== Dataset completo: 541 linhas ===
                                          d.Producto  d.Valor Registrado  \
0                               Ganado Bovino En Pie  4092601413172.4189   
1                             Aceite Crudo De  Palma  2635339077643.2378   
2                     Alimento Concentrado Para Aves  2629196099541.9551   
3      Arroz Blanco Nacional En Bolsa De Polietileno  1906023026699.8291   
4                                      Azucar Blanco   1701074492938.782   
5                   Alimento Concentrado Para Cerdos    1583948004510.15   
6                  Huevo Fresco De Gallina Rojo (Un)   1558380753770.751   
7                                       Cafe Excelso   1552991510722.686   
8  Carne De Cerdo Fresca O Refrigerada En Canal (Kg)   1521889422673.125   
9                            Maiz Amarillo Importado   1193822107225.782   

   d.Año inicial  
0           2019  
1           2010  
2           2010  
3           2010  
4           2010  
5          

### Passo 5 — Download dos arquivos gerados

O pipeline gerou dois arquivos no diretório de trabalho do Colab:

| Arquivo | Conteúdo | Origem |
|---|---|---|
| `precios_2025.csv` | Entrega principal do desafio (2 produtos, layout do PDF) | Passo 3 |
| `dataset_completo.csv` | Dataset completo paginado, sem agregação | Passo 4 |

A célula abaixo dispara o download de ambos diretamente no navegador via
API do Google Colab (`google.colab.files`).

#### Execução fora do Colab

Caso este notebook seja executado em outro ambiente (Jupyter local,
VSCode, etc.), a importação de `google.colab.files` falhará. Os arquivos
permanecem disponíveis no diretório atual do processo Python e podem ser
acessados diretamente pelo sistema de arquivos — o `try/except` abaixo
trata esse caso silenciosamente para que o notebook não quebre em ambientes
alternativos.

In [11]:
# ──────────────────────────────────────────────────────────────────────
# Passo 5: Download dos arquivos gerados
# ──────────────────────────────────────────────────────────────────────
log.info("\n%s", "=" * 60)
log.info("PASSO 5 — Download dos arquivos")
log.info("=" * 60)

ARQUIVOS_GERADOS = ["precios_2025.csv", "dataset_completo.csv"]

try:
    from google.colab import files
    for arquivo in ARQUIVOS_GERADOS:
        log.info("↓ Disparando download: %s", arquivo)
        files.download(arquivo)
    log.info("✓ Downloads iniciados — verifique o navegador")
except ImportError:
    log.info("Ambiente não-Colab detectado. Arquivos disponíveis em:")
    import os
    for arquivo in ARQUIVOS_GERADOS:
        caminho = os.path.abspath(arquivo)
        log.info("  • %s", caminho)

log.info("\n%s", "=" * 60)
log.info("✓ Pipeline completo")
log.info("=" * 60)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>